In [102]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.svm import SVR, SVC
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import cross_val_score, KFold
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel

from scipy.stats import norm
from scipy.optimize import minimize
from sklearn.metrics import r2_score
from scipy.spatial.distance import cdist
from scipy.stats.qmc import LatinHypercube

from skopt.sampler import Lhs
from skopt.space import Space

import shap
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

In [79]:
def compute_ucb(mu, sigma, kappa=2.5):
    return mu + kappa * sigma

def compute_ei(mu, sigma, f_best, xi=0.01):
    z = (mu - f_best - xi) / (sigma + 1e-9)
    ei = (mu - f_best - xi) * norm.cdf(z) + sigma * norm.pdf(z)
    ei[sigma == 0] = 0
    return ei

def format_query(point, decimals=6):
    point_clipped = np.clip(point, 0, 0.999999)
    return '-'.join([f'{x:.{decimals}f}' for x in point_clipped])

def thompson_sample(gp, candidates, n_samples=10, subsample=500):
    idx = np.random.choice(len(candidates), size=min(subsample, len(candidates)), replace=False)
    sub = candidates[idx]
    samples = gp.sample_y(sub, n_samples=n_samples, random_state=42)
    best_sub_idx = np.argmax(samples.mean(axis=1))
    return idx[best_sub_idx]

# Function 1 - week 6

In [104]:
# =============================================================================
# FUNCTION 1 - Radiation Detection (2D)
# changes: 
#     - signal mask replaces log transform -- fit GP only on points with abs(output) > 1e-30
#     - exploiting  gap in the data between "signal" (log_mag > -20) and "dead zone" (log_mag < -39)
#     - candidates increased 10k -> 50k for finer GP resolution
# =============================================================================

print("=" * 60)
print("Function 1 - Week 6")
print("=" * 60)

inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_1/initial_inputs.npy')
outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_1/initial_outputs.npy')

prev_queries = np.array([
    [0.959184, 0.836735],
    [0.374540, 0.950714],
    [0.694651, 0.629916],
    [0.461537, 0.459084],
    [0.618043, 0.460066],
])
prev_outputs = np.array([
    -5.909566597235814e-107,
    -1.560646704467778e-117,
    -0.0016067678433140744,
    -0.000016877758079573665,
    2.602669489913104e-20,
])

all_inputs  = np.vstack([inputs, prev_queries])
all_outputs = np.hstack([outputs, prev_outputs])

print(f"Total points: {len(all_outputs)}")
print(f"Current best: {all_outputs.max():.6e}")

# only fit GP on points with weak or strong signal (ignore extremely tiny values)
# floor to include (week 5 at 2.6e-20) and exclude most everything else
signal_mask = np.abs(all_outputs) > 1e-30
signal_inputs  = all_inputs[signal_mask]
signal_outputs = all_outputs[signal_mask]

print(f"Signal points: {signal_mask.sum()} of {len(all_outputs)}")
print(f"Signal outputs: {signal_outputs}")

kernel = Matern(nu=2.5, length_scale=0.15, length_scale_bounds="fixed")
gp     = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, normalize_y=True)
gp.fit(signal_inputs, signal_outputs)

np.random.seed(42)
candidates = np.random.uniform(0, 1, (50000, 2))  # 50k for finer resolution near peak

mu, sigma = gp.predict(candidates, return_std=True)
ucb       = compute_ucb(mu, sigma, kappa=4.0)

best_idx = np.argmax(ucb)
query    = candidates[best_idx]

best_initial_idx = np.argmax(outputs)
print(f"\nBest initial point: {inputs[best_initial_idx]}, output: {outputs[best_initial_idx]:.6e}")

print(f"\nWeek 6 Query: {format_query(query)}")
print(f"mu: {mu[best_idx]:.4f}, sigma: {sigma[best_idx]:.4f}")

Function 1 - Week 6
Total points: 15
Current best: 7.710875e-16
Signal points: 5 of 15
Signal outputs: [ 7.71087511e-16 -3.60606264e-03 -1.60676784e-03 -1.68777581e-05
  2.60266949e-20]

Best initial point: [0.73102363 0.73299988], output: 7.710875e-16

Week 6 Query: 0.914142-0.732619
mu: 0.0005, sigma: 0.0013


# Function 2 - week 6

In [105]:
# =============================================================================
# FUNCTION 2 - Noisy Log-Likelihood (2D)
# changes: 
#     - Thompson sampling on RF (sample a single tree per candidate for diversity) 
# =============================================================================

print("\n" + "=" * 60)
print("Function 2 - Week 6")
print("=" * 60)

initial_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_2/initial_inputs.npy')
initial_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_2/initial_outputs.npy')

prev_queries = np.array([
    [0.775510, 0.959184],
    [0.683114, 0.932567],
    [0.794441, 0.256481],
    [0.706387, 0.952221],
    [0.693183, 0.938929],
])
prev_outputs = np.array([0.16576674, 0.56974583, 0.27313450, 0.67545988, 0.67430262])

all_inputs  = np.vstack([initial_inputs, prev_queries])
all_outputs = np.hstack([initial_outputs, prev_outputs])

print(f"Total points: {len(all_outputs)}")
print(f"Current best: {all_outputs.max():.3f}")

# RF -- back to max_depth=3, depth=6 showed no benefit
rf = RandomForestRegressor(n_estimators=500, max_depth=3, min_samples_leaf=2, random_state=42)
rf.fit(all_inputs, all_outputs)
print(f"Feature importances: x1={rf.feature_importances_[0]:.3f}, x2={rf.feature_importances_[1]:.3f}")

# Thompson sampling -- draw from single random tree rather than ensemble mean
# leverages RF's internal diversity directly, no kappa to tune
np.random.seed(42)
candidates  = np.random.uniform(0, 1, (50000, 2))
sampled_tree = rf.estimators_[np.random.randint(0, len(rf.estimators_))]
ts_preds     = sampled_tree.predict(candidates)
best_idx     = np.argmax(ts_preds)
query        = candidates[best_idx]

# also compute mean for reference
mean_preds   = np.array([tree.predict(candidates[best_idx:best_idx+1]) 
                for tree in rf.estimators_]).mean()

print(f"\nWeek 6 Query: {format_query(query)}")
print(f"Sampled tree prediction: {ts_preds[best_idx]:.3f}")
print(f"Ensemble mean at query:  {mean_preds:.3f}")


Function 2 - Week 6
Total points: 15
Current best: 0.675
Feature importances: x1=0.768, x2=0.232

Week 6 Query: 0.374540-0.950714
Sampled tree prediction: 0.615
Ensemble mean at query:  0.190


# Function 3 - week 6

In [112]:
# =============================================================================
# FUNCTION 3 - Drug Discovery (3D)
# changes: 
#     - increase max_features to capture interaction terms
#     - increasing max_features gives trees better chance of learning interaction structure
# =============================================================================

print("\n" + "=" * 60)
print("Function 3 - Week 6")
print("=" * 60)

f3_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_3/initial_inputs.npy')
f3_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_3/initial_outputs.npy')

prev_queries = np.array([
    [0.378956, 0.302768, 0.459346],
    [0.315339, 0.088659, 0.415174],
    [0.392735, 0.504381, 0.464332],
    [0.498607, 0.467046, 0.477827],
    [0.432840, 0.535542, 0.476983],
])
prev_outputs = np.array([
    -0.02906213067759293,
    -0.06230989251412482,
    -0.0006328364393800658,
    -0.0029446702316140265,
    -0.0006169444563794807,
])

X = np.vstack([f3_inputs, prev_queries])
Y = np.hstack([f3_outputs, prev_outputs])

print(f"Total points: {len(Y)}, best: {Y.max():.6f}")

# --- Polynomial features (explicit compound interactions) 
poly   = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

feature_names = ['A', 'B', 'C', 'A*B', 'A*C', 'B*C', 'A^2', 'B^2', 'C^2']

# --- RF on polynomial features 
rf = ExtraTreesRegressor(n_estimators=200, max_features='sqrt', min_samples_leaf=2, random_state=42)
rf.fit(X_poly, Y)

print("\nRF feature importances:")
for name, imp in zip(feature_names, rf.feature_importances_):
    print(f"  {name:6s}: {imp:.4f}")

train_preds = rf.predict(X_poly)
print(f"\nRF train R2: {r2_score(Y, train_preds):.4f}")
print(f"RF train predictions vs actual:")
for pred, actual in zip(train_preds, Y):
    print(f"  predicted: {pred:.6f}, actual: {actual:.6f}")

# --- Global candidate search 
np.random.seed(42)
candidates = np.random.uniform(0, 1, (10000, 3))
cand_poly  = poly.transform(candidates)

rf_preds  = rf.predict(cand_poly)
best_idx  = np.argmax(rf_preds)
query     = candidates[best_idx]

top_n = 10
top_indices = np.argsort(rf_preds)[::-1][:top_n]
print(f"\nTop {top_n} RF candidates:")
for i, idx in enumerate(top_indices):
    print(f"  {i+1}: {format_query(candidates[idx])}, RF predicted: {rf_preds[idx]:.6f}")

print(f"\nWeek 6 Query: {format_query(query)}")
print(f"RF predicted: {rf_preds[best_idx]:.6f}")


Function 3 - Week 6
Total points: 20, best: -0.000617

RF feature importances:
  A     : 0.1051
  B     : 0.0480
  C     : 0.2507
  A*B   : 0.0576
  A*C   : 0.0478
  B*C   : 0.0534
  A^2   : 0.0481
  B^2   : 0.1387
  C^2   : 0.2506

RF train R²: 0.6931
RF train predictions vs actual:
  predicted: -0.100653, actual: -0.112122
  predicted: -0.089821, actual: -0.087963
  predicted: -0.096359, actual: -0.111415
  predicted: -0.053698, actual: -0.034835
  predicted: -0.069722, actual: -0.048008
  predicted: -0.083901, actual: -0.110621
  predicted: -0.218168, actual: -0.398926
  predicted: -0.153745, actual: -0.113869
  predicted: -0.103534, actual: -0.131461
  predicted: -0.096784, actual: -0.094190
  predicted: -0.069685, actual: -0.046947
  predicted: -0.084486, actual: -0.105965
  predicted: -0.145301, actual: -0.118048
  predicted: -0.072230, actual: -0.036378
  predicted: -0.083622, actual: -0.056758
  predicted: -0.044274, actual: -0.029062
  predicted: -0.064719, actual: -0.062310


# Function 4 - week 6

In [107]:
# =============================================================================
# FUNCTION 4 - Warehouse Placement (4D)
# changes: 
#     - replaced linear/logistic regression with SVM classifier
#     - candidate generation broadened to all three positive points (w1/w2/w4)
# =============================================================================

print("\n" + "=" * 60)
print("Function 4 - Week 6")
print("=" * 60)

f4_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_4/initial_inputs.npy')
f4_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_4/initial_outputs.npy')

prev_queries = np.array([
    [0.466173, 0.451984, 0.359193, 0.383111],
    [0.424201, 0.406375, 0.372722, 0.413313],
    [0.413541, 0.373697, 0.536536, 0.453354],
    [0.424125, 0.404716, 0.487507, 0.367688],
    [0.488793, 0.294975, 0.332280, 0.395576],
])
prev_outputs = np.array([-0.9654345395220925, 0.6308582112564989, -2.1500998298742817, -0.9915950770116662,
                          -2.4630197287139697])

f4_all_inputs  = np.vstack([f4_inputs, prev_queries])
f4_all_outputs = np.hstack([f4_outputs, prev_outputs])

print(f"Total points: {len(f4_all_outputs)}, best: {f4_all_outputs.max():.4f}")

# SVM Classifier: Non-linear boundary between good and bad regions
# threshold=-1.1 captures W1/W2/W4 as positive (all around -1.0 to +0.6)
# excludes W3/W5 (-2.15, -2.46), which sit closer to the initial data range
# class_weight='balanced' handles remaining imbalance
threshold = -1.1
y_binary = (f4_all_outputs > threshold).astype(int)

print(f"\nSVM Classifier:")
print(f"  Threshold: {threshold}")
print(f"  Above threshold: {y_binary.sum()}/{len(y_binary)}")

svm = SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42)
svm.fit(f4_all_inputs, y_binary)

# Candidate Generation
np.random.seed(42)

week1_query = np.array([0.466173, 0.451984, 0.359193, 0.383111])  # -0.965
week2_query = np.array([0.424201, 0.406375, 0.372722, 0.413313])  # +0.631 best
week4_query = np.array([0.424125, 0.404716, 0.487507, 0.367688])  # -0.992

# local search across all three positive points -- consistent with svm threshold
# weighted toward w2 (best result) but acknowledges the broader good region
# w2: 3500, w1: 2000, w4: 1500 -- proportional to confidence in each centre
local_search = []
for centre, n in [(week2_query, 3500), (week1_query, 2000), (week4_query, 1500)]:
    for _ in range(n):
        candidate = np.clip(centre + np.random.normal(0, 0.05, 4), 0, 1)
        local_search.append(candidate)

# feature-weighted exploration around w2
feature_weighted = []
for _ in range(3000):
    candidate = week2_query.copy()
    candidate += np.random.normal(0, 0.1, 4)
    candidate = np.clip(candidate, 0, 1)
    candidate[2] = np.clip(candidate[2], 0, 0.42)  # x3 asymmetric cap
    candidate[1] = np.clip(candidate[1], 0.38, 1)  # x2 floor
    feature_weighted.append(candidate)

candidates = np.vstack([local_search, feature_weighted])

# SVM filtering: remove candidates the classifier rates as likely bad
# only keep candidates with P(good) > 0.3 -- lenient to avoid over-filtering
svm_probs = svm.predict_proba(candidates)[:, 1]
candidates = candidates[svm_probs > 0.3]
print(f"\nCandidates after SVM filter: {len(candidates)}")

# Gaussian Process -- ARD Matern
# separate length scale per dimension - incorporating what worked and didn't work from previous weeks: 
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(
    length_scale=[0.3, 0.3, 0.2, 0.3],
    length_scale_bounds=[(0.1, 2.0), (0.1, 2.0), (0.1, 0.5), (0.1, 2.0)],  # x3 tight
    nu=2.5
)
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=20, alpha=1e-6, normalize_y=True)
gp.fit(f4_all_inputs, f4_all_outputs)

# Acquisition Function   
mu, sigma = gp.predict(candidates, return_std=True)
ei = compute_ei(mu, sigma, f4_all_outputs.max(), xi=0.01)  # Balanced

best_index = np.argmax(ei)
query = candidates[best_index]

print(f"\nWeek 6 Query: {format_query(query)}")
print(f"mean: {mu[best_index]:.3f} \nstandard deviation: {sigma[best_index]:.3f}")
print(f"EI: {ei[best_index]:.6f}")


Function 4 - Week 6
Total points: 35, best: 0.6309

SVM Classifier:
  Threshold: -1.1
  Above threshold: 3/35

Candidates after SVM filter: 9061

Week 6 Query: 0.303872-0.408521-0.392275-0.469543
mean: 1.595 
standard deviation: 0.746
EI: 0.989956


# Function 5 - week 6

In [108]:
# =============================================================================
# FUNCTION 5 - Chemical Yield (4D)
# changes from w5:
#   - added w5 data point
#   - SVR candidate pre-filter lowered from 3000 to 500 to be more selective
#   - dimensional pinning adjusted:
#        x4 pinned at 0.999999 (saturated in w5, same logic as x3)
#       directed: x1 now UP (data shows consistent positive gradient on x1)
#       directed: x4 push removed (now pinned)
#   - top-5 EI diagnostic added
# =============================================================================

print("\n" + "=" * 60)
print("Function 5 - Week 6")
print("=" * 60)

f5_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_5/initial_inputs.npy')
f5_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_5/initial_outputs.npy')

prev_queries = np.array([
    [0.284290, 0.869208, 0.999999, 0.903273],  # week 1: 2201.83
    [0.311208, 0.881577, 0.999999, 0.915050],  # week 2: 2381.54
    [0.355085, 0.899160, 0.999900, 0.934196],  # week 3: 2689.15
    [0.453609, 0.922659, 0.999999, 0.960267],  # week 4: 3223.24
    [0.493503, 0.872392, 0.999999, 0.999999],  # week 5: 3286.99
])
prev_outputs = np.array([2201.834589108927, 2381.536867607932, 2689.1537294933396, 3223.2410694936825, 3286.9929500236235])

all_inputs  = np.vstack([f5_inputs, prev_queries])
all_outputs = np.hstack([f5_outputs, prev_outputs])

print(f"Total points: {len(all_outputs)}, best so far: {all_outputs.max():.2f}")

# SVR: StandardScaler on raw outputs (unchanged)
input_scaler  = StandardScaler()
output_scaler = StandardScaler()
X_scaled = input_scaler.fit_transform(all_inputs)
Y_scaled = output_scaler.fit_transform(all_outputs.reshape(-1, 1)).ravel()

svr = SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1)
svr.fit(X_scaled, Y_scaled)

# GP: log-transform outputs (unchanged)
y_log     = np.log1p(all_outputs)
kernel_gp = ConstantKernel(1.0, (0.001, 1000)) * Matern(length_scale=0.2, nu=2.5)
gp        = GaussianProcessRegressor(kernel=kernel_gp, n_restarts_optimizer=25, normalize_y=True)
gp.fit(all_inputs, y_log)

best_point = all_inputs[np.argmax(all_outputs)]
np.random.seed(42)

# local: small perturbations around best point (unchanged except x4 pinned)
local = []
for _ in range(6000):
    c    = best_point + np.random.normal(0, 0.03, 4)
    c    = np.clip(c, 0, 1)
    c[2] = 0.999999  # x3 pinned
    c[3] = 0.999999  # x4 pinned (new)
    local.append(c)

# directed: x1 up (corrected from w5), x2 symmetric, x4 pinned
directed = []
for _ in range(4000):
    c    = best_point.copy()
    c[0] = np.clip(c[0] + abs(np.random.normal(0, 0.05)), 0, 1)  
    c[1] = np.clip(c[1] + np.random.normal(0, 0.03), 0, 1)        # x2 symmetric (no assumed direction)
    c[2] = 0.999999
    c[3] = 0.999999                                               # x4 pinned (was being pushed up)
    directed.append(c)

candidates = np.vstack([local, directed])

# SVR pre-filter: keep top 500
X_cand_scaled = input_scaler.transform(candidates)
svr_preds_s   = svr.predict(X_cand_scaled)
svr_preds     = output_scaler.inverse_transform(svr_preds_s.reshape(-1, 1)).ravel()

top_idx  = np.argsort(svr_preds)[-500:] # lowered from 3000 so that SVR is more selective
filtered = candidates[top_idx]

# GP + EI on log scale (unchanged)
mu_log, sigma_log = gp.predict(filtered, return_std=True)
ei                = compute_ei(mu_log, sigma_log, np.log1p(all_outputs.max()), xi=0.005)

best_idx = np.argmax(ei)
query    = filtered[best_idx]

print(f"\nWeek 6 Query:       {format_query(query)}")
print(f"GP predicted yield: {np.expm1(mu_log[best_idx]):.2f}")
print(f"GP std (log scale): {sigma_log[best_idx]:.4f}")
print(f"SVR prediction:     {svr_preds[top_idx[best_idx]]:.2f}")
print(f"EI:                 {ei[best_idx]:.6f}")

# Diagnostic: sanity-check GP/SVR agreement and ridge width
top5_ei = np.argsort(ei)[-5:][::-1]
print("\nTop-5 EI candidates:")
for rank, i in enumerate(top5_ei):
    print(f"  #{rank+1}: x={format_query(filtered[i])}  GP={np.expm1(mu_log[i]):.1f}  SVR={svr_preds[top_idx[i]]:.1f}  EI={ei[i]:.6f}")


Function 5 - Week 6
Total points: 25, best so far: 3286.99

Week 6 Query:       0.520874-0.972838-0.999999-0.999999
GP predicted yield: 3888.34
GP std (log scale): 0.2361
SVR prediction:     3221.12
EI:                 0.197254

Top-5 EI candidates:
  #1: x=0.520874-0.972838-0.999999-0.999999  GP=3888.3  SVR=3221.1  EI=0.197254
  #2: x=0.520052-0.968853-0.999999-0.999999  GP=3870.5  SVR=3223.3  EI=0.190485
  #3: x=0.514845-0.965779-0.999999-0.999999  GP=3862.6  SVR=3225.4  EI=0.185844
  #4: x=0.500933-0.962806-0.999999-0.999999  GP=3861.2  SVR=3223.4  EI=0.182384
  #5: x=0.498759-0.961950-0.999999-0.999999  GP=3858.1  SVR=3222.5  EI=0.181108


# Function 6 - week 6

In [109]:
# =============================================================================
# FUNCTION 6 - Cake Recipe (5D)
# change: 
#     - replacing single-tree Thompson sampling with full-ensemble UCB
#     - replace random uniform candidates with Maximin LHS candidate generation
#     - RF surrogate retained (increased n_estimators to 500 to get more reliable uncertainty estimates)
#     - sampling maximin LHS maximises (minimum distance between candidates, ensuring UCB evaluates a well-spread pool)
# =============================================================================

print("\n" + "=" * 60)
print("Function 6 - Week 6")
print("=" * 60)

f6_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_6/initial_inputs.npy')
f6_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_6/initial_outputs.npy')

prev_queries = np.array([
    [0.201818, 0.220639, 0.476414, 0.867793, 0.023115],  # W1: y=-0.792246
    [0.457382, 0.349799, 0.529520, 0.691032, 0.192589],  # W2: y=-0.361637
    [0.503404, 0.328861, 0.662838, 0.995031, 0.025514],  # W3: y=-0.367626
    [0.484266, 0.300747, 0.678836, 0.776480, 0.244496],  # W4: y=-0.314817 (best)
    [0.341066, 0.113474, 0.924694, 0.877339, 0.257942],  # W5: y=-0.828981
])
prev_outputs = np.array([-0.792246, -0.361637, -0.367626, -0.314817, -0.828981])

all_inputs  = np.vstack([f6_inputs, prev_queries])
all_outputs = np.hstack([f6_outputs, prev_outputs])

print(f"Total observations: {all_inputs.shape[0]}, best so far: {all_outputs.max():.4f}")

# Random Forest surrogate - retained from W5
# increased n_estimators to 500 to get more reliable uncertainty estimates
rf = RandomForestRegressor(n_estimators=500, random_state=42)
rf.fit(all_inputs, all_outputs)

print("\nRandom Forest feature importances:")
for i, importance in enumerate(rf.feature_importances_):
    print(f"  x{i+1}: {importance:.4f}")

# Candidate generation - Maximin LHS replaces random uniform
space      = Space([(0.0, 1.0)] * 5)
lhs        = Lhs(criterion="maximin", iterations=100)
candidates = np.array(lhs.generate(space.dimensions, 5000))

# Full-ensemble UCB acquisition
# collect predictions from every tree to compute mean and std across the forest
# std measures tree disagreement - high where the model is uncertain
kappa      = 1.5
tree_preds = np.array([t.predict(candidates) for t in rf.estimators_])
mean_pred  = tree_preds.mean(axis=0)
std_pred   = tree_preds.std(axis=0)
ucb_scores = mean_pred + kappa * std_pred

best_index = np.argmax(ucb_scores)
query      = candidates[best_index]

print(f"\nWeek 6 Query: {format_query(query)}")
print(f"  Predicted mean: {mean_pred[best_index]:.4f}")
print(f"  Predicted std:  {std_pred[best_index]:.4f}")
print(f"  UCB score:      {ucb_scores[best_index]:.4f}  (kappa={kappa})")


Function 6 - Week 6
Total observations: 25, best so far: -0.3148

Random Forest feature importances:
  x1: 0.0650
  x2: 0.1427
  x3: 0.0504
  x4: 0.4242
  x5: 0.3176

Week 6 Query: 0.896209-0.283983-0.594508-0.806739-0.008612
  Predicted mean: -0.7155
  Predicted std:  0.4366
  UCB score:      -0.0606  (kappa=1.5)


# Functon 7 - week 6

In [110]:
# =============================================================================
# FUNCTION 7 - ML Hyperparameters (6D)
# changes: 
#     - restored RF importance from W3 (dropped SHAP analysis)
#     - bounds set directly from observed results
#     - EI replaced with UCB (kappa = 2.0) for explicit exploration
# =============================================================================

print("\n" + "=" * 60)
print("Function 7 - Week 6")
print("=" * 60)

f7_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_7/initial_inputs.npy')
f7_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_7/initial_outputs.npy')

prev_queries = np.array([
    [0.045091, 0.528666, 0.329265, 0.105350, 0.434671, 0.641164],  # w1: 1.051
    [0.034389, 0.427542, 0.236649, 0.342956, 0.405706, 0.695527],  # w2: 1.653
    [0.020000, 0.062635, 0.587368, 0.313225, 0.362008, 0.664539],  # w3: 2.602 (best)
    [0.012096, 0.033240, 0.640160, 0.336533, 0.245422, 0.901043],  # w4: 1.509
    [0.007872, 0.174935, 0.499616, 0.265730, 0.340533, 0.150339],  # w5: 0.383
])
prev_outputs = np.array([1.0510006614196026, 1.6531363312716738, 2.6016443512251484, 1.5087286481808686, 0.3832671270292543])

all_inputs  = np.vstack([f7_inputs, prev_queries])
all_outputs = np.hstack([f7_outputs, prev_outputs])

print(f"Total points: {len(all_outputs)}, best: {all_outputs.max():.4f}")

# Step 1: RF importance
print("\nStep 1: Random Forest")
feature_names = [f"HP{i+1}" for i in range(6)]

random_forest = RandomForestRegressor(n_estimators=200, random_state=42)
random_forest.fit(all_inputs, all_outputs)

print("\nRF Feature Importance:")
for name, importance in zip(feature_names, random_forest.feature_importances_):
    print(f"  {name}: {importance:.3f}")


# Step 2: Subspace definition
# window width set by RF importance — high importance, narrow window.
# placement set from observed weekly results
print("\nStep 2: Subspace Definition")

importance    = random_forest.feature_importances_
width         = 1.0 - importance

# HP2-HP5 more room to change - keep HP1 pushed high, and HP6 at observed sweet spot in previous weeks
subspace_lower = np.array([0.02, 0.0, 0.0, 0.0, 0.0, 0.60])
subspace_upper = np.array([width[0], 1.0, 1.0, 1.0, 1.0, 0.85])

subspace_lower = np.clip(subspace_lower, 0, 1)
subspace_upper = np.clip(subspace_upper, 0, 1)

print("\nSubspace bounds:")
for name, low, high, imp in zip(feature_names, subspace_lower, subspace_upper, importance):
    print(f"  {name}: [{low:.3f}, {high:.3f}]  (importance {imp:.3f})")


# Step 3: GP with UCB inside the subspace
# Outputs log-transformed to stabilise GP fit across wide output range.
print("\nStep 3: GP with UCB")

kernel = ConstantKernel(1.0, (0.01, 10.0)) * Matern(
    length_scale=[0.3] * 6,
    length_scale_bounds=(0.05, 5.0),
    nu=2.5,
)
gp = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=20,
    normalize_y=True,
)
gp.fit(all_inputs, np.log1p(all_outputs))


def negative_ucb(candidate, gp, kappa=2.0):
    candidate = candidate.reshape(1, -1)
    mean, std = gp.predict(candidate, return_std=True)
    return -(mean[0] + kappa * std[0])


bounds = list(zip(subspace_lower, subspace_upper))

best_ucb_value = np.inf
best_candidate = None

np.random.seed(42)
for _ in range(500):
    start = np.array([np.random.uniform(low, high) for low, high in bounds])
    result = minimize(
        negative_ucb,
        start,
        args=(gp,),
        bounds=bounds,
        method="L-BFGS-B",
    )
    if result.fun < best_ucb_value:
        best_ucb_value = result.fun
        best_candidate = result.x

predicted_mean, predicted_std = gp.predict(best_candidate.reshape(1, -1), return_std=True)
predicted_y = np.expm1(predicted_mean[0])

print(f"\nWeek 6 Query: {format_query(best_candidate)}")
print(f"\nGP predicted Y (original scale) : {predicted_y:.4f}")
print(f"GP uncertainty (log scale std)  : {predicted_std[0]:.4f}")


Function 7 - Week 6
Total points: 35, best: 2.6016

Step 1: Random Forest

RF Feature Importance:
  HP1: 0.676
  HP2: 0.066
  HP3: 0.039
  HP4: 0.041
  HP5: 0.060
  HP6: 0.118

Step 2: Subspace Definition

Subspace bounds:
  HP1: [0.020, 0.324]  (importance 0.676)
  HP2: [0.000, 1.000]  (importance 0.066)
  HP3: [0.000, 1.000]  (importance 0.039)
  HP4: [0.000, 1.000]  (importance 0.041)
  HP5: [0.000, 1.000]  (importance 0.060)
  HP6: [0.600, 0.850]  (importance 0.118)

Step 3: GP with UCB

Week 6 Query: 0.020000-0.000000-0.000000-0.212520-0.346351-0.696182

GP predicted Y (original scale) : 2.4267
GP uncertainty (log scale std)  : 0.0749


# Function 8 - week 6

In [113]:
# =============================================================================
# FUNCTION 8 - 8D Black-Box
# changes: 
#    - MLP dropout_rate changed from 0.1 to 0.3 and hidden_dim from 32 to 16 (less overfitting),
#    - kappa lowered from 3.0 to 1.7 
#    - added LHS candidate generation, changed from np.random.uniform 
# =============================================================================

print("\n" + "=" * 60)
print("Function 8 - Week 6")
print("=" * 60)

import torch
import torch.nn as nn
from scipy.stats.qmc import LatinHypercube

f8_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_8/initial_inputs.npy')
f8_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_8/initial_outputs.npy')

prev_queries = np.array([
    [0.182943, 0.000000, 0.000000, 0.000000, 0.999999, 0.127124, 0.092403, 0.000000],  # W1: 9.6724
    [0.022695, 0.000000, 0.274348, 0.000000, 0.999999, 0.127124, 0.028034, 0.000000],  # W2: 9.6264
    [0.005086, 0.000000, 0.048567, 0.000000, 0.999999, 0.998962, 0.018674, 0.000000],  # W3: 9.5464
    [0.000000, 0.000000, 0.012582, 0.000000, 0.999999, 0.000000, 0.066411, 0.000000],  # W4: 9.5519
    [0.143046, 0.934478, 0.256793, 0.378273, 0.985471, 0.743394, 0.090143, 0.014829],  # W5: 9.1457
])
prev_outputs = np.array([9.6723503773075, 9.6263579169495, 9.5463675507445, 9.5519471979855, 9.1457303782294])

all_inputs  = np.vstack([f8_inputs, prev_queries])
all_outputs = np.hstack([f8_outputs, prev_outputs])

print(f"Total points: {len(all_outputs)}, best: {all_outputs.max():.4f} (W1)")

# scale inputs and outputs
scaler_x = StandardScaler()
scaler_y = StandardScaler()
X_sc = scaler_x.fit_transform(all_inputs)
Y_sc = scaler_y.fit_transform(all_outputs.reshape(-1, 1)).ravel()

X_tensor = torch.tensor(X_sc, dtype=torch.float32)
Y_tensor = torch.tensor(Y_sc, dtype=torch.float32).unsqueeze(1)

# MLP with dropout
# dropout stays active during prediction for MC uncertainty estimates
class DropoutMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout_rate):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.net(x)


# LOO CV - calibration check on the torch DropoutMLP
loo_errors = []
loo_sigmas = []

for i in range(len(all_outputs)):
    # hold out point i
    X_train_loo = np.delete(all_inputs, i, axis=0)
    y_train_loo = np.delete(all_outputs, i)
    X_test_loo  = all_inputs[i:i+1]
    y_test_loo  = all_outputs[i]

    # fit scalers on training fold only -- no leakage
    sx_loo = StandardScaler()
    sy_loo = StandardScaler()
    X_tr_sc_loo = sx_loo.fit_transform(X_train_loo)
    y_tr_sc_loo = sy_loo.fit_transform(y_train_loo.reshape(-1, 1)).ravel()
    X_te_sc_loo = sx_loo.transform(X_test_loo)

    X_tr_tensor = torch.tensor(X_tr_sc_loo, dtype=torch.float32)
    Y_tr_tensor = torch.tensor(y_tr_sc_loo, dtype=torch.float32).unsqueeze(1)
    X_te_tensor = torch.tensor(X_te_sc_loo, dtype=torch.float32)

    # fresh model for each fold
    torch.manual_seed(42)
    model_loo     = DropoutMLP(input_dim=8, hidden_dim=24, dropout_rate=0.2)
    optimiser_loo = torch.optim.Adam(model_loo.parameters(), lr=1e-3)

    model_loo.train()
    for epoch in range(1000):
        optimiser_loo.zero_grad()
        loss_loo = nn.MSELoss()(model_loo(X_tr_tensor), Y_tr_tensor)
        loss_loo.backward()
        optimiser_loo.step()

    # MC dropout prediction on held-out point
    mu_sc_loo, sigma_sc_loo = mc_predict(model_loo, X_te_tensor, n_passes=50)
    mu_loo    = sy_loo.inverse_transform(mu_sc_loo.reshape(-1, 1)).ravel()[0]
    sigma_loo = np.atleast_1d(sigma_sc_loo)[0] * sy_loo.scale_[0]
    
    loo_errors.append(abs(mu_loo - y_test_loo))
    loo_sigmas.append(sigma_loo)

loo_errors = np.array(loo_errors)
loo_sigmas = np.array(loo_sigmas)

print(f"\nLOO Calibration (true torch model):")
print(f"  mean |error|:        {loo_errors.mean():.4f}")
print(f"  std  |error|:        {loo_errors.std():.4f}")
print(f"  max  |error|:        {loo_errors.max():.4f}")
print(f"  mean sigma (LOO):    {loo_sigmas.mean():.4f}")
print(f"  ratio (error/sigma): {loo_errors.mean() / loo_sigmas.mean():.3f}  (1.0 = perfect calibration)")
print(f"  suggested kappa:     {kappa * (loo_errors.mean() / loo_sigmas.mean()):.2f}")


torch.manual_seed(42)
model     = DropoutMLP(input_dim=8, hidden_dim=24, dropout_rate=0.2)  # W5: hidden=32, dropout=0.1
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn   = nn.MSELoss()

# train
model.train()
for epoch in range(1000):
    optimiser.zero_grad()
    pred = model(X_tensor)
    loss = loss_fn(pred, Y_tensor)
    loss.backward()
    optimiser.step()

print(f"Training loss (final): {loss.item():.4f}")

# MC dropout prediction -- keep model in train() so dropout stays active
def mc_predict(model, X_tensor, n_passes=50):
    model.train()
    preds = torch.stack([model(X_tensor).squeeze() for _ in range(n_passes)])
    return preds.mean(dim=0).detach().numpy(), preds.std(dim=0).detach().numpy()

# LHS candidate pool 
n    = 50000
pool = LatinHypercube(d=8, seed=42).random(n=n)

pool_sc     = scaler_x.transform(pool)
pool_tensor = torch.tensor(pool_sc, dtype=torch.float32)

mu_sc, sigma_sc = mc_predict(model, pool_tensor, n_passes=50)

# inverse transform mu back to original scale for interpretability
mu    = scaler_y.inverse_transform(mu_sc.reshape(-1, 1)).ravel()
sigma = sigma_sc * scaler_y.scale_[0]

# UCB acquisition
kappa = 1.7  # W5: 3.0 
ucb   = mu + kappa * sigma
best_idx = np.argmax(ucb)
query    = pool[best_idx]

print(f"mu   range: {mu.min():.3f} to {mu.max():.3f}")
print(f"sigma range: {sigma.min():.3f} to {sigma.max():.3f}")
print(f"kappa*sigma range: {(kappa*sigma).min():.3f} to {(kappa*sigma).max():.3f}")

print(f"\nWeek 6 Query: {format_query(query)}")
print(f"MC mu: {mu[best_idx]:.4f}, sigma: {sigma[best_idx]:.4f}")
print(f"UCB:   {ucb[best_idx]:.4f}")


Function 8 - Week 6
Total points: 45, best: 9.6724 (W1)

LOO Calibration (true torch model):
  mean |error|:        0.2767
  std  |error|:        0.2279
  max  |error|:        1.0071
  mean sigma (LOO):    0.2439
  ratio (error/sigma): 1.135  (1.0 = perfect calibration)
  suggested kappa:     1.93
Training loss (final): 0.0417
mu   range: 5.049 to 9.695
sigma range: 0.061 to 0.721
kappa*sigma range: 0.104 to 1.226

Week 6 Query: 0.060495-0.027090-0.006185-0.132068-0.160719-0.354431-0.001361-0.931902
MC mu: 9.6950, sigma: 0.3521
UCB:   10.2935
